# Data Preprocessing (Interactive Notebook)

This notebook runs the repository's Step1→Step9 data-preprocessing steps interactively so you can execute, inspect and debug individual stages locally.

Outputs from these steps will be written under `example/data_preprocessing` and downstream under `example/machine_learning` by default.

In [ ]:
%%bash
git clone https://github.com/hlnicholls/GenePrioritiser
cd GenePrioritiser

conda env create -f GenePrioritiser_env.yml
conda activate GenePrioritiser_env
pip install --force-reinstall scikit-learn==1.4.2
pip install --force-reinstall scipy==1.11.4
pip install --force-reinstall numpy==1.23.0
pip install scikit-optimize
conda install -c bioconda bedtools htslib
conda install -c conda-forge parallel
pip install pybedtools intervaltree requests

In [1]:
import os
os.chdir("/Users/hannahnicholls/GitHub/GenePrioritiser")

In [2]:
!bash /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step1_annotate_genes_bedtools.sh

Using cached gene bed: /Users/hannahnicholls/GitHub/GenePrioritiser/utils/gencode_v19.genes.ext10000.sorted.bed
Processing /Users/hannahnicholls/GitHub/GenePrioritiser/./example/data_preprocessing/input/Evangelou_30224653_DBP.txt.gz in /Users/hannahnicholls/GitHub/GenePrioritiser/tmp_annotation_Evangelou_30224653_DBP.txt_27680
Parsing GWAS input and extracting SNP positions (Python)...
Extracted 7160657 SNP positions from /Users/hannahnicholls/GitHub/GenePrioritiser/./example/data_preprocessing/input/Evangelou_30224653_DBP.txt.gz
Running bedtools closest (this is the fast step, but can be I/O heavy on large GWAS)...
Building gene map (line -> gene, distance)...
Joining original rows with gene map and writing outputs via Python...
Wrote annotated 7160657 SNPs to ./example/data_preprocessing/output/variants/Annotated_GWAS_DBP.csv
Wrote variant data to ./example/data_preprocessing/output/variants/variant_data_DBP.csv
Cleaning up temporary files: /Users/hannahnicholls/GitHub/GenePrioritise

In [3]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step2_harmonise_genes.py

Repo root         : /Users/hannahnicholls/GitHub/GenePrioritiser
Variant directory : example/data_preprocessing/output/variants
HGNC file         : /Users/hannahnicholls/GitHub/GenePrioritiser/utils/hgnc_complete_set.txt
Building HGNC synonym map...
Loaded 102319 synonym mappings

=== Harmonising Annotated_GWAS_* files ===
  Processing Annotated_GWAS_DBP.csv ... [OK, 428054 gene names changed]
  Processing Annotated_GWAS_PP.csv ... [OK, 423689 gene names changed]
  Processing Annotated_GWAS_SBP.csv ... [OK, 423690 gene names changed]

=== Harmonising variant_data_* files ===
  Processing variant_data_DBP.csv ... [OK, 428054 gene names changed]
  Processing variant_data_PP.csv ... [OK, 423689 gene names changed]
  Processing variant_data_SBP.csv ... [OK, 423690 gene names changed]

Done. All applicable files overwritten with HGNC-harmonised Gene symbols.


In [4]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step3_process_variant_level_data.py

All files processed and merged. Output saved to: ./databases/variant_level/merged_gene_median_variant_measures.csv


In [5]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step4_least_likely_gene_selection.py

Number of unmapped entries: 0
Sample unmapped entries:
Empty DataFrame
Columns: [protein1, protein2, coexpression, experiments, database, Gene1, Gene2]
Index: []
Interactors collected with max_depth=3: 19430 genes
Direct only interactors (depth=1): 17554
One-level interactors (depth=2): 19428
Found 3 annotated GWAS file(s) to load:
 - ./example/data_preprocessing/output/variants/Annotated_GWAS_DBP.csv
 - ./example/data_preprocessing/output/variants/Annotated_GWAS_PP.csv
 - ./example/data_preprocessing/output/variants/Annotated_GWAS_SBP.csv
Wrote intermediate least-likely genes to /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/least_likely_intermediate.tsv (will not overwrite ./example/data_preprocessing/input/least_likely_genes.tsv)
Applying additional filtering based on: ./example/data_preprocessing/input/BP_loci_Apr2020_LDr2-8_500kb.csv
Filtered out 65 genes based on additional criteria (no genes with SNPs in any LD with any BP loci).
Updated i

In [6]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step6_identify_training_genes.py

Using intermediate least-likely genes file: /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/least_likely_intermediate.tsv
Identified 13758 genes with at least one P<0.01 SNP in EACH Annotated_GWAS file (per-file intersection)
Probable genes: 254 -> 145 after applying per-file P<0.01 intersection filter


In [7]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step7_merge_all_databases_and_get_training_data.py

Found 5116 duplicate Gene entries in merged data - aggregating numeric features by mean to collapse duplicates.
All merged databases saved to ./example/data_preprocessing/output/all_genes_merged_all_data.csv
Training genes counts per label:
  least likely: 571
  probable: 145
  most likely: 51
Joined data saved to ['./example/data_preprocessing/output/training_data_all_features.csv', './example/machine_learning/eda/input/training_data_all_features.csv']


In [8]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step8_downsample_least_likely_genes.py

Using intermediate least-likely genes file: /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/least_likely_intermediate.tsv
/Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step8_downsample_least_likely_genes.py:311: UserWarning: The palette list has more values (2) than needed (1), which may not be intended.
  sns.boxplot(data=[combined_pos['variant_count'], least_df['variant_count']], palette=['C0', 'C1'])
Wrote diagnostic plots: /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/variant_count_hist_seed42.png /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/variant_count_box_seed42.png
Wrote updated training_genes to ./example/data_preprocessing/input/training_genes.txt
Wrote sampled least file: /Users/hannahnicholls/GitHub/GenePrioritiser/example/data_preprocessing/input/sampling/least_likely_sampled_seed42.tsv
Wrote audit: /Users/hannahnicholls/GitHub/Ge

In [9]:
!python /Users/hannahnicholls/GitHub/GenePrioritiser/src/data_preprocessing/Step9_subset_genes_to_prioritise.py